# Quarry similarity search

Find 10 m pixels across Fiji whose [OlmoEarth v1.3 embeddings](https://source.coop/olmoearth/olmoearth-v1-3-embeddings) are similar to the quarry training points.

1. Sample the embedding at each quarry point.
2. For every pixel, score = max cosine similarity to any quarry point.
3. Map the 10 m scores masked to >= 90%, over satellite imagery, with the quarry points. Nothing is saved to disk.

In [ ]:
import numpy as np
import xarray as xr
from dask.distributed import Client as DaskClient
from odc.geo.geom import box
from odc.geo.xr import crop

from utils import BBOX, CHUNKS, training_data

In [ ]:
# BBOX from utils. West of 180° Fiji is all UTM zone 60; east of 180° (Lau group) is zone 1, not done here.
YEAR = 2024  # OlmoEarth is global for 2023-2025
QUARRY = 8  # lulc_code
NODATA = -128
STRIP = 128  # rows scored at once: (STRIP x 2048 x n_quarry) float32 per thread
OLMOEARTH_URL = "s3://olmoearth/olmoearth-v1-3-embeddings/embeddings.zarr"

In [ ]:
dask_client = DaskClient(n_workers=1, threads_per_worker=16, memory_limit="16GB")

## Open embeddings

Consolidated metadata: one request to open. `chunks=None`: lazy without a dask graph over the whole UTM zone; dask chunks are added after cropping. Only bands 0-63 (a complete 64-d Matryoshka embedding, half the download).

In [ ]:
ds = xr.open_zarr(
    OLMOEARTH_URL,
    group="utm60",  # TODO: Hardcoded. Need to loosen. Could be good for performance I guess.
    storage_options={"anon": True, "endpoint_url": "https://data.source.coop"},
    chunks=None,  # TODO: Consider setting chunks for performance?
    zarr_format=3,
    consolidated=True,
)
emb = (
    ds["embeddings"]
    .sel(time=YEAR)
    .isel(band=slice(0, 64))
    .odc.assign_crs(ds.attrs["proj:code"])
)
emb = crop(emb, box(*BBOX, crs="EPSG:4326"), apply_mask=False)
emb

In [ ]:
def normalise(q: np.ndarray) -> np.ndarray:
    """int8 embeddings (..., band) -> unit vectors; nodata -> NaN."""
    q = q.astype("float32")
    v = (np.abs(q) / 127.5) ** 2 * np.sign(q)  # signed-power de-quantization
    v[q == NODATA] = np.nan
    return v / np.linalg.norm(v, axis=-1, keepdims=True)

## Quarry embeddings

In [ ]:
points = training_data.query("lulc_code == @QUARRY").to_crs(emb.odc.crs)
xy = points.get_coordinates()
samples = emb.sel(
    x=xr.DataArray(xy.x.values, dims="point"),
    y=xr.DataArray(xy.y.values, dims="point"),
    method="nearest",
).transpose("point", "band")
quarry = normalise(samples.values)
quarry = quarry[~np.isnan(quarry).any(axis=1)]
print(f"{len(quarry)} of {len(points)} quarry points have embeddings")

In [ ]:
# How alike are quarries? Each point's best match among the *other* points.
# Points in the same quarry match ~1, so this is optimistic; use it to pick a threshold.
sim = quarry @ quarry.T
np.fill_diagonal(sim, -1)
print(np.percentile(sim.max(axis=1), [5, 25, 50]).round(3))

## Score every pixel

In [ ]:
def max_similarity(block: np.ndarray) -> np.ndarray:
    """(y, x, band) int8 -> (y, x) uint8 max cosine similarity x 100; 255 = nodata.

    Scored in STRIP-row strips to cap memory; all-nodata (ocean) strips are skipped.
    """
    out = np.full(block.shape[:-1], 255, "uint8")
    for i in range(0, len(block), STRIP):
        b = block[i : i + STRIP]
        if (b == NODATA).all():
            continue
        s = np.nanmax(normalise(b) @ quarry.T, axis=-1, initial=-1)
        out[i : i + STRIP] = np.where(s == -1, 255, np.clip(s, 0, 1) * 100)
    return out


score = xr.apply_ufunc(
    max_similarity,
    emb.chunk(band=-1, **CHUNKS),  # 2048 = the store's shard size
    input_core_dims=[["band"]],
    dask="parallelized",
    output_dtypes=["uint8"],
)
score.attrs["nodata"] = 255
score

In [ ]:
THRESHOLD = 60  # % cosine similarity
# THRESHOLD = 90  # % cosine similarity

In [ ]:
# Write the results to a COG
from odc.geo.xr import write_cog

write_cog(
    score,
    "score.tif",
    overwrite=True,
)

## Map

In [ ]:
m = score.where((score >= THRESHOLD) & (score <= 100)).odc.explore(
    cmap="Spectral",
    vmin=THRESHOLD,
    vmax=100,
    tiles="Esri.WorldImagery",
    name=f">= {THRESHOLD}% similar",
)
points.explore(m=m, color="cyan", name="quarry training points")
m